In [0]:
%spark.pyspark from pyspark.sql import functions as F, Window
users = spark.createDataFrame(
    [ ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg"), ],
    ["user_id", "city"] 
)
orders = spark.createDataFrame(
    [ ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0), ],
    ["order_id", "user_id", "product_id", "qty", "price"] 
)
products = spark.createDataFrame(
    [ ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA"), ],
    ["product_id", "product_name"] 
)

In [1]:
%spark.pyspark
win = Window.partitionBy("city", "product_id", "product_name")
top_2_win = Window.partitionBy("city").orderBy("revenue_sum")

In [2]:
%spark.pyspark
mart_city_top_products = users \
    .join(orders, on="user_id", how="inner") \
    .join(products, on="product_id", how="inner") \
    .withColumn("revenue", F.col("qty") * F.col("price")) \
    .withColumn("orders_cnt", F.count("*").over(win)) \
    .withColumn("qty_sum", F.sum("qty").over(win)) \
    .withColumn("revenue_sum", F.sum("revenue").over(win)) \
    .withColumn("top_revenuesum_by_city", F.dense_rank().over(top_2_win)) \
    .filter(F.col("top_revenuesum_by_city") == 2)

In [3]:
%spark.pyspark
mart_city_top_products.show(10)

In [4]:
%spark.pyspark
mart_city_top_products.write \
    .mode("overwrite") \
    .parquet("hdfs:///user/zeppelin/tmp/sandbox_zeppelin/mart_city_top_products")

In [5]:
%spark.pyspark
mart_city_top_products.write \
    .mode("overwrite") \
    .parquet("s3a://apache/tmp/sandbox_zeppelin/mart_city_top_products")

In [6]:
%spark.pyspark

df = spark.read.parquet("hdfs:///user/zeppelin/tmp/sandbox_zeppelin/mart_city_top_products")

df.show(10)

In [7]:
%spark.pyspark
df = spark.read.parquet("s3a://apache/tmp/sandbox_zeppelin/mart_city_top_products")
z.show(df)

In [8]:
%spark.pyspark
